In [1]:
import pandas as pd
import json
from collections import Counter

# Load data
with open("../crawl_metadata/metadata_preprocessed.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)


In [2]:

# Xem unique category
categories = df['category'].dropna().tolist()

counter = Counter(categories)

# Top category phổ biến
for cat, count in counter.items():
    print(cat, ":", count)

Nhà hàng : 112
Nhà hàng điểm tâm : 2
Nhà hàng Việt Nam : 12
Quán mì : 13
Nhà hàng món nướng Hàn Quốc : 2
Thẩm mỹ viện : 1
Quán cà phê : 13
Quán ăn nhỏ : 72
Nhà hàng cho người ăn chay : 8
Nhà hàng hải sản : 5
Nhà hàng Nhật Bản : 5
Nhà hàng Trung Quốc : 8
Nhà hàng món lẩu : 9
Nhà hàng thịt gà : 7
Trung tâm đào tạo : 1
Cửa hàng bán đồ ăn nấu sẵn : 2
Nhà hàng châu Á : 11
Nhà hàng Hàn Quốc : 7
Nhà bán buôn sản phẩm làm đẹp : 1
Nhà hàng gia đình : 6
Cửa hàng thịt gà : 1
Nhà hàng cơm : 7
Điểm phát chẩn đồ ăn : 1
Gian hàng rong : 2
Nhà hàng ăn nhanh : 5
Trà trân châu : 3
Nhà hàng phở : 14
Quán ăn tự phục vụ : 1
Nhà hàng chuyên phục vụ bữa sáng : 2
Nhà hàng món nướng : 3
Nhà hàng ăn chay : 1
Tiệm chăm sóc móng : 1
Nhà hàng bít tết : 1
Nhà hàng món kem mứt : 1
Món nướng : 2
Cửa hàng : 1
Nhà hàng Thái Lan : 3
Nhà cung cấp thực phẩm và đồ uống cho dịch vụ ăn uống : 1
Khu ăn uống : 1
Cửa hàng kem : 2
Cửa hàng quần áo : 1
Quán súp : 1
Nhà hàng món chay : 3
Quán ăn : 2
Cửa hàng mỹ phẩm : 2
Nhà hàn

In [3]:
import pandas as pd
import unicodedata

# ===== 1. Normalize =====
def normalize_text(text):
    if text is None:
        return ""
    return unicodedata.normalize("NFC", text).lower().strip()


# ===== 2. Mapping =====
def map_category(cat):
    cat = normalize_text(cat)

    # ===== CHỈ GIỮ những cái liên quan tới ăn uống =====
    VALID_HINT = [
        "nhà hàng", "quán", "mì", "phở", "bún", "cơm",
        "lẩu", "nướng", "ăn", "gian hàng", "cửa hàng"
    ]
    if not any(k in cat for k in VALID_HINT):
        return None

    # ===== NHÓM QUỐC GIA =====
    if "hàn quốc" in cat:
        return "Nhà hàng Hàn Quốc"

    if "việt nam" in cat:
        return "Nhà hàng Việt Nam"

    if "nhật bản" in cat:
        return "Nhà hàng Nhật Bản"

    if "trung quốc" in cat or "quảng đông" in cat:
        return "Nhà hàng Trung Quốc"

    if "thái lan" in cat:
        return "Nhà hàng Thái Lan"

    # ===== QUÁN ĂN NHỎ (bạn định nghĩa cứng) =====
    SMALL_EAT_KEYWORDS = [
        "món nướng",
        "quán súp", "quán sup",
        "quán ăn",
        "mang về",
        "ăn nhanh",
        "cửa hàng ăn nhanh",
        "cửa hàng bánh hấp",
        "cửa hàng bán đồ tráng miệng",
        "quán ăn nhẹ",
        "tự phục vụ",
        "gian hàng rong",
        "đồ ăn nấu sẵn",
        "quán mì",
        "mì"
    ]

    if any(k in cat for k in SMALL_EAT_KEYWORDS):
        return "Quán ăn nhỏ"

    # ===== NHÀ HÀNG CHUNG =====
    if "nhà hàng" in cat:
        return "Nhà hàng"

    # ===== còn lại bỏ =====
    return None


# ===== 3. Apply =====
def clean_category_df(df):
    df = df.copy()

    df["category_clean"] = df["category"].apply(map_category)

    # drop hết cái không hợp lệ
    df_clean = df[df["category_clean"].notnull()].copy()

    return df_clean


# ===== 4. RUN =====
df_clean = clean_category_df(df)

print("Before:", len(df))
print("After:", len(df_clean))

print("\n=== CATEGORY CLEAN ===")
print(df_clean["category_clean"].value_counts())

Before: 394
After: 349

=== CATEGORY CLEAN ===
category_clean
Nhà hàng               197
Quán ăn nhỏ            111
Nhà hàng Việt Nam       12
Nhà hàng Trung Quốc     12
Nhà hàng Hàn Quốc        9
Nhà hàng Nhật Bản        5
Nhà hàng Thái Lan        3
Name: count, dtype: int64


# Xử lý normalize dữ liệu name

In [4]:
import unicodedata

def normalize_text(text):
    if pd.isna(text):
        return ""
    return unicodedata.normalize("NFC", str(text)).strip()

# apply toàn bộ cột text quan trọng
for col in ["name", "category", "address"]:
    df_clean[col] = df_clean[col].apply(normalize_text)

In [5]:
df_clean[df_clean["name"].str.contains("Ốc Nhi", na=False)]

,url,name,avg_rating,review_count,category,price_range,address,phone,opening_hours,lgbtq_friendly,popular_times,url_source,about,category_clean
94,https://www.google.com/maps/place/%E1%BB%90c+N...,Ốc Nhi 2N,"3,8",564,Nhà hàng,₫₫,"80 Nguyễn Biểu, phường 01, Chợ Quán, Hồ Chí Mi...",+84 908 553 030,"{'Thứ Sáu': '16:00–04:00', 'Thứ Bảy': '16:00–0...",có,"{'Chủ Nhật': ['Mức độ đông là 0% lúc 06 giờ.',...",https://www.google.com/maps/place/%E1%BB%90c+N...,{'phu_hop_cho_nguoi_khuyet_tat': ['Chỗ đỗ xe c...,Nhà hàng


In [6]:
from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse
import hashlib
import pandas as pd

def canonicalize_google_maps_url(url):
    if pd.isna(url) or str(url).strip() == "":
        return ""

    url = str(url).strip()

    try:
        parsed = urlparse(url)

        drop_params = {"hl", "authuser", "rclk"}

        query_pairs = parse_qsl(parsed.query, keep_blank_values=True)
        filtered_pairs = [(k, v) for k, v in query_pairs if k not in drop_params]
        new_query = urlencode(filtered_pairs)

        canonical = urlunparse((
            parsed.scheme.lower(),
            parsed.netloc.lower(),
            parsed.path,
            parsed.params,
            new_query,
            ""
        ))

        return canonical
    except Exception:
        return url

def make_restaurant_id(url):
    canonical_url = canonicalize_google_maps_url(url)
    if canonical_url == "":
        return None
    return hashlib.md5(canonical_url.encode("utf-8")).hexdigest()

df_clean["source_url_for_id"] = df_clean["url"].fillna("")

if "url_source" in df_clean.columns:
    df_clean["source_url_for_id"] = df_clean["source_url_for_id"].mask(
        df_clean["source_url_for_id"].eq(""),
        df_clean["url_source"]
    )

df_clean["canonical_url"] = df_clean["source_url_for_id"].apply(canonicalize_google_maps_url)
df_clean["restaurant_id"] = df_clean["source_url_for_id"].apply(make_restaurant_id)

print(df_clean[["name", "url", "url_source", "canonical_url", "restaurant_id"]].head())

                              name  \
0                  Quán ǎn Hùng Ký   
1                 Nhà hàng Văn Hoa   
2   Tiến Phát - Điểm tâm Hồng Kông   
3  Bò Nướng Phố - Nguyễn Chí Thanh   
4                 Bún Thang Cậu Ba   

                                                 url  \
0  https://www.google.com/maps/place/Qu%C3%A1n+%C...   
1  https://www.google.com/maps/place/Van+Hoa+Rest...   
2  https://www.google.com/maps/place/Ti%E1%BA%BFn...   
3  https://www.google.com/maps/place/B%C3%B2+N%C6...   
4  https://www.google.com/maps/place/B%C3%BAn+Tha...   

                                          url_source  \
0  https://www.google.com/maps/place/Qu%C3%A1n+%C...   
1  https://www.google.com/maps/place/Van+Hoa+Rest...   
2  https://www.google.com/maps/place/Ti%E1%BA%BFn...   
3  https://www.google.com/maps/place/B%C3%B2+N%C6...   
4  https://www.google.com/maps/place/B%C3%BAn+Tha...   

                                       canonical_url  \
0  https://www.google.com/maps/place/Qu%C

# Chuẩn hóa avg_rating, review_count, price_range

In [7]:
# ===== CHECK MISSING =====
cols = ["avg_rating", "review_count", "price_range"]

print("=== MISSING VALUES ===")
for col in cols:
    missing = df_clean[col].isnull().sum()
    total = len(df_clean)
    print(f"{col}: {missing}/{total} ({missing/total:.2%})")

=== MISSING VALUES ===
avg_rating: 0/349 (0.00%)
review_count: 0/349 (0.00%)
price_range: 108/349 (30.95%)


In [8]:
print("\n=== SAMPLE VALUES ===")
for col in cols:
    print(f"\n{col}:")
    print(df_clean[col].dropna().unique()[:10])


=== SAMPLE VALUES ===

avg_rating:
['3,3' '4,3' '4,4' '4,2' '4,6' '4,0' '4,8' '3,9' '3,5' '4,1']

review_count:
['102' '801' '1850' '1188' '55' '623' '159' '679' '155' '210']

price_range:
['100-200 N ₫' '1-100.000 ₫' '200-300 N ₫' '₫₫' '300-400 N ₫'
 'Phở trộn cực ngon, ớt tương chuẩn vị bắc, 10₫, giá rất rẻ so với mặt bằng chung trong khu vực trung tâm'
 '₫' '₫₫₫']


price_range có dữ liệu rác, sẽ xóa dữ liệu rác và tiến hành tiền xử lý các phân khúc giá lại cho đồng bộ

In [9]:
import re
import pandas as pd

def clean_price_range(p):
    if pd.isna(p):
        return None, None, None
    
    p = str(p).lower().strip()

    # ===== CASE 1: ₫₫ =====
    if re.fullmatch(r"₫+", p):
        return None, None, len(p)

    # ===== CLEAN TEXT =====
    p = p.replace("n ₫", "").replace("₫", "").strip()

    # ===== CASE 2: range =====
    match = re.search(r"(\d[\d\.]*)\s*-\s*(\d[\d\.]*)", p)
    
    if match:
        try:
            low = int(match.group(1).replace(".", ""))
            high = int(match.group(2).replace(".", ""))

            # normalize nếu số nhỏ (100 → 100k)
            if low < 1000:
                low *= 1000
            if high < 1000:
                high *= 1000

            avg = (low + high) / 2

            if avg < 50000:
                level = 1
            elif avg < 150000:
                level = 2
            else:
                level = 3

            return low, high, level
        except:
            return None, None, None

    return None, None, None

df_clean[["price_min", "price_max", "price_level"]] = df_clean["price_range"].apply(
    lambda x: pd.Series(clean_price_range(x))
)

print("Parsed price:", df_clean["price_min"].notna().sum())
print("Parsed level:", df_clean["price_level"].notna().sum())


fail_cases = df_clean[
    df_clean["price_range"].notna() &
    df_clean["price_min"].isna() &
    df_clean["price_level"].isna()
]

print(len(fail_cases))
print(fail_cases["price_range"].head(20))

Parsed price: 214
Parsed level: 240
1
90    Phở trộn cực ngon, ớt tương chuẩn vị bắc, 10₫,...
Name: price_range, dtype: object


In [10]:
# ===== FORMAT ĐẸP =====
for i, row in df_clean.head(30).iterrows():
    print(f"--- {row['name']} ---")
    print("raw:   ", row["price_range"])
    print("clean: ", row["price_min"], "-", row["price_max"])
    print("level: ", row["price_level"])
    print()



--- Quán ǎn Hùng Ký ---
raw:    100-200 N ₫
clean:  100000.0 - 200000.0
level:  3.0

--- Nhà hàng Văn Hoa ---
raw:    None
clean:  nan - nan
level:  nan

--- Tiến Phát - Điểm tâm Hồng Kông ---
raw:    100-200 N ₫
clean:  100000.0 - 200000.0
level:  3.0

--- Bò Nướng Phố - Nguyễn Chí Thanh ---
raw:    100-200 N ₫
clean:  100000.0 - 200000.0
level:  3.0

--- Bún Thang Cậu Ba ---
raw:    1-100.000 ₫
clean:  1000.0 - 100000.0
level:  2.0

--- Hải Ký Mì Gia ---
raw:    100-200 N ₫
clean:  100000.0 - 200000.0
level:  3.0

--- GoGi House Hùng Vương Plaza ---
raw:    200-300 N ₫
clean:  200000.0 - 300000.0
level:  3.0

--- GÓC HUẾ - An Dương Vương ---
raw:    100-200 N ₫
clean:  100000.0 - 200000.0
level:  3.0

--- Chấn Hưng Nem Nướng ---
raw:    1-100.000 ₫
clean:  1000.0 - 100000.0
level:  2.0

--- Bún đậu mắm tôm Tiến Hải-Chi nhánh 6 ---
raw:    1-100.000 ₫
clean:  1000.0 - 100000.0
level:  2.0

--- Lẩu cua Đất Mũi ---
raw:    ₫₫
clean:  nan - nan
level:  2.0

--- Lẩu Cá Dân Ích ---
raw:   

In [11]:
def clean_rating(x):
    if pd.isna(x):
        return None
    try:
        return float(str(x).replace(",", "."))
    except:
        return None

df_clean["rating"] = df_clean["avg_rating"].apply(clean_rating)
df_clean[["avg_rating", "rating"]].head(10)

,avg_rating,rating
0,"3,3",3.3
1,"4,3",4.3
2,"4,4",4.4
3,"4,2",4.2
4,"4,6",4.6
5,"4,0",4.0
6,"4,2",4.2
8,"4,8",4.8
9,"3,9",3.9
10,"3,5",3.5


# Xử lý biến address, phone, lgbtq_friendly
Không chỉnh sửa gì vì đã phù hợp

# Xử lý dữ liệu opening_hour
- qua quan sát thấy có nhiều quán bị cào thiếu thứ 6, xử lý và cào lại trước khi thật sự tiền xử lý dữ liệu opening hour

In [12]:
missing_friday = []

for _, row in df_clean.iterrows():
    oh = row.get("opening_hours", {})
    
    if not isinstance(oh, dict) or "Thứ Sáu" not in oh:
        url = row.get("url")
        if url:
            missing_friday.append(url)

print("Missing Friday:", len(missing_friday))

Missing Friday: 52


In [13]:
with open("missing_friday.txt", "w", encoding="utf-8") as f:
    for url in missing_friday:
        f.write(url + "\n")

In [14]:
missing_friday[:5]

['https://www.google.com/maps/place/Nem+Nuong/data=!4m7!3m6!1s0x31752fcaed1ff91b:0x87e52aa522239334!8m2!3d10.7525695!4d106.6739911!16s%2Fg%2F11hjxpllqq!19sChIJG_kf7covdTERNJMjIqUq5Yc?authuser=0&hl=vi&rclk=1',
 'https://www.google.com/maps/place/Rin+Sushi/data=!4m7!3m6!1s0x31752f50c2d8898b:0x6fa9f70c8b188314!8m2!3d10.758304!4d106.6826312!16s%2Fg%2F11h16nt6p4!19sChIJi4nYwlAvdTERFIMYiwz3qW8?authuser=0&hl=vi&rclk=1',
 'https://www.google.com/maps/place/H%E1%BB%A7+ti%E1%BA%BFu+nam+vang+Ph%C6%B0%E1%BB%A3ng/data=!4m7!3m6!1s0x31752efb6ec40471:0x2b3726cafa6a6678!8m2!3d10.7534294!4d106.6690772!16s%2Fg%2F11c1tt2dcw!19sChIJcQTEbvsudTEReGZq-somNys?authuser=0&hl=vi&rclk=1',
 'https://www.google.com/maps/place/L%E1%BA%A9u+B%C3%B2+H%C3%A2n/data=!4m7!3m6!1s0x31752f00758be9b9:0x8d7198cef7368658!8m2!3d10.7554216!4d106.6839786!16s%2Fg%2F11y96gyz9b!19sChIJuemLdQAvdTERWIY2986YcY0?authuser=0&hl=vi&rclk=1',
 'https://www.google.com/maps/place/B%C3%BAn+Th%E1%BB%8Bt+N%C6%B0%E1%BB%9Bng+H%E1%BA%B1ng/data=!4m7!3m6

Đã cào lại những quán bị thiếu opening_hour, tiến hành matching cho đầy đủ dữ liệu hơn

In [15]:
import json

# ===== 1. Load data mới =====
with open("metadata_missing_friday.json", "r", encoding="utf-8") as f:
    new_data = json.load(f)

# ===== 2. Normalize URL (tránh lệch) =====
def normalize_url(url):
    if not isinstance(url, str):
        return url
    return url.split("?")[0]

# ===== 3. Build lookup =====
new_opening_map = {}

for item in new_data:
    url = normalize_url(item.get("url"))
    opening = item.get("opening_hours")
    
    if url and isinstance(opening, dict) and len(opening) > 0:
        new_opening_map[url] = opening


# ===== 4. Update df_clean =====
updated_count = 0
still_missing = []

for idx, row in df_clean.iterrows():
    url = normalize_url(row.get("url"))
    oh = row.get("opening_hours")

    # ===== CASE LỖI =====
    is_empty = not isinstance(oh, dict) or len(oh) == 0
    missing_friday = isinstance(oh, dict) and "Thứ Sáu" not in oh

    if is_empty or missing_friday:
        if url in new_opening_map:
            df_clean.at[idx, "opening_hours"] = new_opening_map[url]
            updated_count += 1
        else:
            still_missing.append(url)

print("Updated:", updated_count)
print("Still missing:", len(still_missing))

Updated: 4
Still missing: 48


In [16]:
missing_after = df_clean[
    df_clean["opening_hours"].apply(
        lambda x: not isinstance(x, dict) or "Thứ Sáu" not in x
    )
]

print("Missing after update:", len(missing_after))

Missing after update: 48


In [17]:
df_clean.loc[
    df_clean["url"].isin(list(new_opening_map.keys())[:5]),
    ["name", "opening_hours"]
]

,name,opening_hours


In [18]:
bad_new = []

for item in new_data:
    oh = item.get("opening_hours")

    if not isinstance(oh, dict) or len(oh) == 0 or "Thứ Sáu" not in oh:
        bad_new.append(item.get("url"))

print("Bad in new_data:", len(bad_new))

Bad in new_data: 49


In [19]:
import re
from collections import defaultdict

def classify_opening_format(value):
    if not value:
        return "empty"
    
    value = str(value).strip().lower()

    # ===== CASE CHUẨN =====
    pattern = re.compile(r"^(\d{2}:\d{2}–\d{2}:\d{2})(\s+\d{2}:\d{2}–\d{2}:\d{2})*$")
    if pattern.fullmatch(value):
        return "normal_time"

    # ===== CASE ĐẶC BIỆT =====
    if "đóng cửa" in value:
        return "closed"
    
    if "mở cửa cả ngày" in value:
        return "open_24h"

    return "other"


# ===== APPLY =====
format_counter = defaultdict(int)
bad_examples = []

for idx, row in df_clean.iterrows():
    oh = row.get("opening_hours", {})
    
    if not isinstance(oh, dict):
        continue

    for day, val in oh.items():
        fmt = classify_opening_format(val)
        format_counter[fmt] += 1

        if fmt == "other":
            bad_examples.append((idx, row.get("name", ""), day, val))


# ===== RESULT =====
print("=== FORMAT COUNT ===")
for k, v in format_counter.items():
    print(f"{k}: {v}")

print("\n=== SAMPLE BAD CASES ===")
for x in bad_examples[:10]:
    print("\n---")
    print("Index:", x[0])
    print("Name:", x[1])
    print("Day:", x[2])
    print("Value:", x[3])

=== FORMAT COUNT ===
normal_time: 2015
closed: 22
open_24h: 70

=== SAMPLE BAD CASES ===


In [20]:
import re

# ===== 1. MAP DAY =====
DAY_MAP = {
    "Thứ Hai": "mon",
    "Thứ Ba": "tue",
    "Thứ Tư": "wed",
    "Thứ Năm": "thu",
    "Thứ Sáu": "fri",
    "Thứ Bảy": "sat",
    "Chủ Nhật": "sun"
}

# ===== 2. PARSE 1 BLOCK TIME =====
def parse_time_block(block):
    match = re.match(r"(\d{2}):(\d{2})–(\d{2}):(\d{2})", block)
    if not match:
        return None
    h1, m1, h2, m2 = map(int, match.groups())
    return (h1, m1, h2, m2)


# ===== 3. PARSE OPENING HOURS =====
def parse_opening_hours(oh):
    if not isinstance(oh, dict) or len(oh) == 0:
        return None

    result = {}

    for day_vi, value in oh.items():
        day_en = DAY_MAP.get(day_vi)
        if not day_en:
            continue

        value = str(value).lower().strip()

        # ===== ĐÓNG CỬA =====
        if "đóng cửa" in value:
            result[day_en] = []
            continue

        # ===== MỞ 24H =====
        if "mở cửa cả ngày" in value:
            result[day_en] = [(0, 0, 23, 59)]
            continue

        # ===== NORMAL =====
        blocks = value.split()
        intervals = []

        for b in blocks:
            parsed = parse_time_block(b)
            if parsed:
                intervals.append(parsed)

        result[day_en] = intervals

    return result if result else None
# ===== 4. APPLY =====
df_clean["opening_parsed"] = df_clean["opening_hours"].apply(parse_opening_hours)


# ===== 5. HÀM CHECK QUÁN CÓ MỞ KHÔNG =====
def is_open(opening_parsed, day, hour, minute):
    if opening_parsed is None:
        return True  # fallback: không biết thì vẫn cho qua

    if day not in opening_parsed:
        return False

    intervals = opening_parsed[day]

    # ===== ĐÓNG CỬA =====
    if len(intervals) == 0:
        return False

    now = hour * 60 + minute

    for (h1, m1, h2, m2) in intervals:
        start = h1 * 60 + m1
        end = h2 * 60 + m2

        # ===== QUA NGÀY =====
        if end < start:
            if now >= start or now <= end:
                return True
        else:
            if start <= now <= end:
                return True

    return False


# ===== 6. DEBUG =====
print("Done parsing opening_hours!")

print("\nSample:")
print(df_clean[["opening_hours", "opening_parsed"]].head(5))

Done parsing opening_hours!

Sample:
                                       opening_hours  \
0  {'Thứ Bảy': '17:00–23:00', 'Chủ Nhật': '17:00–...   
1  {'Thứ Bảy': '07:00–13:30 17:00–21:30', 'Chủ Nh...   
2  {'Thứ Bảy': '06:00–12:30', 'Chủ Nhật': '06:00–...   
3  {'Thứ Bảy': '16:30–22:00', 'Chủ Nhật': '16:30–...   
4  {'Thứ Bảy': '06:30–14:00 18:00–21:30', 'Chủ Nh...   

                                      opening_parsed  
0  {'sat': [(17, 0, 23, 0)], 'sun': [(17, 0, 23, ...  
1  {'sat': [(7, 0, 13, 30), (17, 0, 21, 30)], 'su...  
2  {'sat': [(6, 0, 12, 30)], 'sun': [(6, 0, 12, 3...  
3  {'sat': [(16, 30, 22, 0)], 'sun': [(16, 30, 22...  
4  {'sat': [(6, 30, 14, 0), (18, 0, 21, 30)], 'su...  


# Xử lý dữ liệu popular_time

Check xem có dòng nào khác format hay thấy trong file không

In [21]:
import re

def check_popular_format(pt):
    if not isinstance(pt, dict):
        return []
    
    bad_rows = []
    
    pattern = re.compile(r"Mức độ đông là (\d+)% lúc (\d{2}) giờ")

    for day, arr in pt.items():
        if not isinstance(arr, list):
            continue
        
        for s in arr:
            s_clean = str(s).strip().replace(".", "")
            
            if not pattern.fullmatch(s_clean):
                bad_rows.append((day, s))
    
    return bad_rows


# ===== APPLY =====
all_bad = []

for idx, row in df_clean.iterrows():
    bad = check_popular_format(row["popular_times"])
    
    if bad:
        all_bad.append({
            "index": idx,
            "name": row.get("name", ""),
            "bad": bad[:3]  # chỉ lấy vài cái cho dễ xem
        })

print("Total bad cases:", len(all_bad))

# in thử 5 cái
for item in all_bad[:5]:
    print("\n---")
    print(item["index"], item["name"])
    print(item["bad"])

Total bad cases: 2

---
94 Ốc Nhi 2N
[('Thứ Sáu', 'Hiện mức độ đông là 59%, thường mức độ đông là 21%.')]

---
277 Phong Vị 24h | Nhà Hàng Lẩu Quận 5
[('Thứ Bảy', 'Hiện mức độ đông là 14%, thường mức độ đông là 8%.')]


In [22]:
def parse_popular_times(pt):
    if not isinstance(pt, dict) or len(pt) == 0:
        return None

    result = {}

    pattern_hour = re.compile(r"(\d+)%.*?(\d{2}) giờ")
    pattern_avg = re.compile(r"thường mức độ đông là (\d+)%")

    for day_vi, arr in pt.items():
        day_en = DAY_MAP.get(day_vi)
        if not day_en:
            continue
        
        day_dict = {}

        fallback_avg = None

        for s in arr:
            s_clean = str(s).replace(".", "").lower()

            # ===== CASE CÓ GIỜ =====
            match = pattern_hour.search(s_clean)
            if match:
                percent = int(match.group(1))
                hour = int(match.group(2))
                day_dict[hour] = percent
                continue

            # ===== CASE "THƯỜNG" =====
            match_avg = pattern_avg.search(s_clean)
            if match_avg:
                fallback_avg = int(match_avg.group(1))

        # ===== FILL nếu thiếu giờ =====
        if fallback_avg is not None:
            for h in range(24):
                if h not in day_dict:
                    day_dict[h] = fallback_avg

        if day_dict:
            result[day_en] = day_dict

    return result if result else None


df_clean["popular_parsed"] = df_clean["popular_times"].apply(parse_popular_times)
df_clean[["popular_times", "popular_parsed"]].head(3)


,popular_times,popular_parsed
0,{},None
1,"{'Chủ Nhật': ['Mức độ đông là 0% lúc 06 giờ.',...","{'sun': {6: 0, 7: 24, 8: 41, 9: 50, 10: 49, 11..."
2,"{'Chủ Nhật': ['Mức độ đông là 0% lúc 05 giờ.',...","{'sun': {5: 0, 6: 29, 7: 57, 8: 84, 9: 100, 10..."


In [23]:
def debug_popular(df, keyword):
    sub = df[df["name"].str.contains(keyword, case=False, na=False)]

    print(f"Found {len(sub)} rows\n")

    for idx, row in sub.iterrows():
        print("=" * 60)
        print("Name:", row["name"])
        print("URL:", row.get("url", ""))

        print("\n--- RAW popular_times ---")
        print(row.get("popular_times"))

        print("\n--- PARSED popular_parsed ---")
        print(row.get("popular_parsed"))

        print("\n")


# Ốc Nhi
debug_popular(df_clean, "Ốc Nhi 2N")

# Quán nướng / Phong Vị 24h
debug_popular(df_clean, "Phong Vị 24h")

Found 1 rows

Name: Ốc Nhi 2N
URL: https://www.google.com/maps/place/%E1%BB%90c+Nhi+2N/data=!4m7!3m6!1s0x31752f04f1c77997:0xcb35b94ab52739c3!8m2!3d10.7550466!4d106.6838758!16s%2Fg%2F11bwfjdhlj!19sChIJl3nH8QQvdTERwzkntUq5Ncs?authuser=0&hl=vi&rclk=1

--- RAW popular_times ---
{'Chủ Nhật': ['Mức độ đông là 0% lúc 06 giờ.', 'Mức độ đông là 0% lúc 07 giờ.', 'Mức độ đông là 0% lúc 08 giờ.', 'Mức độ đông là 0% lúc 09 giờ.', 'Mức độ đông là 0% lúc 10 giờ.', 'Mức độ đông là 0% lúc 11 giờ.', 'Mức độ đông là 0% lúc 12 giờ.', 'Mức độ đông là 0% lúc 13 giờ.', 'Mức độ đông là 0% lúc 14 giờ.', 'Mức độ đông là 0% lúc 15 giờ.', 'Mức độ đông là 21% lúc 16 giờ.', 'Mức độ đông là 33% lúc 17 giờ.', 'Mức độ đông là 52% lúc 18 giờ.', 'Mức độ đông là 65% lúc 19 giờ.', 'Mức độ đông là 73% lúc 20 giờ.', 'Mức độ đông là 73% lúc 21 giờ.', 'Mức độ đông là 71% lúc 22 giờ.', 'Mức độ đông là 63% lúc 23 giờ.', 'Mức độ đông là 48% lúc 00 giờ.', 'Mức độ đông là 36% lúc 01 giờ.', 'Mức độ đông là 27% lúc 02 giờ.', 'Mức độ

# Xử lý mục About

các danh mục trong about hiện tại chưa có 1 list cố định cứng, nên eda để trích ra

In [24]:
from collections import Counter

# ===== 1. Lấy tất cả keys =====
all_keys = []

# ===== 2. Lấy tất cả values theo từng key =====
all_values = {}

for about in df_clean["about"]:
    if isinstance(about, dict):
        for k, v in about.items():
            all_keys.append(k)

            if k not in all_values:
                all_values[k] = []

            if isinstance(v, list):
                all_values[k].extend(v)

# ===== 3. Đếm key =====
key_counter = Counter(all_keys)

print("=== ABOUT KEYS ===")
for k, c in key_counter.most_common():
    print(f"{k}: {c}")

# ===== 4. Unique values mỗi key =====
print("\n=== UNIQUE VALUES PER KEY ===")

for k, values in all_values.items():
    unique_vals = list(set(values))
    print(f"\n--- {k} ({len(unique_vals)} values) ---")
    print(unique_vals[:50])  # in 50 cái đầu cho dễ nhìn

=== ABOUT KEYS ===
cac_tuy_chon_dich_vu: 233
lua_chon_an_uong: 228
bau_khong_khi: 215
tien_nghi: 194
khach_hang: 194
dich_vu: 169
noi_tieng_ve: 158
bai_do_xe: 158
len_ke_hoach: 123
phu_hop_cho_nguoi_khuyet_tat: 122
thanh_toan: 118
tre_em: 112
diem_noi_bat: 67
tu_doanh_nghiep: 14
thu_cung: 14

=== UNIQUE VALUES PER KEY ===

--- phu_hop_cho_nguoi_khuyet_tat (5 values) ---
['Chỗ đỗ xe cho xe lăn', 'Lối vào cho xe lăn', 'Nhà vệ sinh cho xe lăn', 'Vòng trợ thính', 'Chỗ ngồi cho xe lăn']

--- cac_tuy_chon_dich_vu (7 values) ---
['Chỗ ngồi ngoài trời', 'Mua hàng ngay trên xe', 'Dịch vụ tại chỗ', 'Ăn tại chỗ', 'Giao hàng gián tiếp', 'Giao hàng', 'Đồ ăn mang đi']

--- diem_noi_bat (10 values) ---
['Nhiều bia ngon', 'Nhiều trà ngon', 'Nhiều rượu vang ngon', 'Chỗ ngồi trên sân thượng', 'Nhạc sống', 'Thể thao', 'Cà phê ngon', 'Lò sưởi', 'Rượu cốc tai ngon', 'Món tráng miệng ngon']

--- noi_tieng_ve (4 values) ---
['Bữa trưa', 'Bữa tối', 'Ăn tối một mình', 'Bữa sáng']

--- dich_vu (20 values) ---
[

Chuẩn hóa để mỗi nhà hàng đều có các mục tiện ích như nhau, có tên mục, value có thể rỗng.

In [25]:
import unicodedata

# ===== 1. Danh sách key cố định =====
ABOUT_KEYS = [
    "cac_tuy_chon_dich_vu",
    "lua_chon_an_uong",
    "bau_khong_khi",
    "tien_nghi",
    "khach_hang",
    "dich_vu",
    "noi_tieng_ve",
    "bai_do_xe",
    "len_ke_hoach",
    "phu_hop_cho_nguoi_khuyet_tat",
    "thanh_toan",
    "tre_em",
    "diem_noi_bat",
    "tu_doanh_nghiep",
    "thu_cung"
]

# ===== 2. Normalize text =====
def normalize_text(text):
    if not text:
        return ""
    return unicodedata.normalize("NFC", text).strip()

# ===== 3. Clean 1 about =====
def clean_about(about):
    clean = {}

    # nếu không phải dict → trả về dict rỗng đủ key
    if not isinstance(about, dict):
        return {k: [] for k in ABOUT_KEYS}

    for key in ABOUT_KEYS:
        values = about.get(key, [])

        # đảm bảo luôn là list
        if not isinstance(values, list):
            values = []

        # normalize + remove rác
        cleaned_vals = []
        for v in values:
            v_clean = normalize_text(v)
            if v_clean:  # bỏ empty string
                cleaned_vals.append(v_clean)

        # remove duplicate
        clean[key] = list(set(cleaned_vals))

    return clean


# ===== 4. Apply vào DataFrame =====
df_clean["about_clean"] = df_clean["about"].apply(clean_about)

print("Done clean about!")

# ===== 5. Check sample =====
print("\n=== SAMPLE ===")
print(df_clean[["about", "about_clean"]].head(3))

Done clean about!

=== SAMPLE ===
                                               about  \
0                                                NaN   
1  {'phu_hop_cho_nguoi_khuyet_tat': ['Chỗ ngồi ch...   
2  {'phu_hop_cho_nguoi_khuyet_tat': ['Chỗ đỗ xe c...   

                                         about_clean  
0  {'cac_tuy_chon_dich_vu': [], 'lua_chon_an_uong...  
1  {'cac_tuy_chon_dich_vu': ['Ăn tại chỗ', 'Đồ ăn...  
2  {'cac_tuy_chon_dich_vu': ['Dịch vụ tại chỗ', '...  


In [26]:
import json

row = df_clean.iloc[1]

print("Name:", row.get("name", "N/A"))
print("URL:", row.get("url", ""))

print("\nABOUT CLEAN:")
print(json.dumps(row["about_clean"], indent=2, ensure_ascii=False))

Name: Nhà hàng Văn Hoa
URL: https://www.google.com/maps/place/Van+Hoa+Restaurant/data=!4m7!3m6!1s0x31752f5dfe64ff03:0x9fcd311260c552f!8m2!3d10.7526983!4d106.6644903!16s%2Fg%2F11fjvv4ydk!19sChIJA_9k_l0vdTERL1UMJhHT_Ak?authuser=0&hl=vi&rclk=1

ABOUT CLEAN:
{
  "cac_tuy_chon_dich_vu": [
    "Ăn tại chỗ",
    "Đồ ăn mang đi"
  ],
  "lua_chon_an_uong": [
    "Bữa nửa buổi",
    "Phục vụ tại bàn",
    "Món tráng miệng",
    "Bữa tối",
    "Bữa trưa",
    "Dịch vụ ăn uống",
    "Bữa sáng",
    "Chỗ ngồi"
  ],
  "bau_khong_khi": [
    "Cao cấp",
    "Lãng mạn",
    "Ấm cúng"
  ],
  "tien_nghi": [
    "Nhà vệ sinh"
  ],
  "khach_hang": [
    "Nhóm",
    "Du khách"
  ],
  "dich_vu": [
    "Rượu",
    "Cà phê",
    "Phòng ăn riêng tư",
    "Rượu cốc tai",
    "Rượu vang",
    "Nhà hàng thức ăn nhanh",
    "Bia",
    "Rượu mạnh",
    "Phần ăn nhỏ"
  ],
  "noi_tieng_ve": [
    "Bữa trưa",
    "Bữa tối",
    "Ăn tối một mình",
    "Bữa sáng"
  ],
  "bai_do_xe": [
    "Đỗ xe có tính phí trên đường",


Check data metadata sau khi tiền xử lý xong

In [27]:
import json

def print_sample_restaurants(df, n=5):
    for i in range(n):
        print("="*80)
        print(f"Restaurant {i}")
        print("="*80)
        
        row = df.iloc[i].to_dict()
        
        # pretty print JSON
        print(json.dumps(row, indent=2, ensure_ascii=False))


# chạy
print_sample_restaurants(df_clean, 5)

Restaurant 0
{
  "url": "https://www.google.com/maps/place/Qu%C3%A1n+%C7%8En+H%C3%B9ng+K%C3%BD/data=!4m7!3m6!1s0x31752efa64fcabf7:0x455fbbc7d3651107!8m2!3d10.7524705!4d106.6658192!16s%2Fg%2F11gdgxn5tq!19sChIJ96v8ZPoudTERBxFl08e7X0U?authuser=0&hl=vi&rclk=1",
  "name": "Quán ǎn Hùng Ký",
  "avg_rating": "3,3",
  "review_count": "102",
  "category": "Nhà hàng",
  "price_range": "100-200 N ₫",
  "address": "Trần Hưng Đạo/218 B, Phường 11, Chợ Lớn, Hồ Chí Minh, Việt Nam",
  "phone": "+84 918 038 796",
  "opening_hours": {
    "Thứ Bảy": "17:00–23:00",
    "Chủ Nhật": "17:00–23:00",
    "Thứ Hai": "Đóng cửa",
    "Thứ Ba": "17:00–23:00",
    "Thứ Tư": "17:00–23:00",
    "Thứ Năm": "17:00–23:00",
    "Thứ Sáu": "17:00–23:00"
  },
  "lgbtq_friendly": "unknown",
  "popular_times": {},
  "url_source": "https://www.google.com/maps/place/Qu%C3%A1n+%C7%8En+H%C3%B9ng+K%C3%BD/data=!4m7!3m6!1s0x31752efa64fcabf7:0x455fbbc7d3651107!8m2!3d10.7524705!4d106.6658192!16s%2Fg%2F11gdgxn5tq!19sChIJ96v8ZPoudTERB

In [28]:
import json
import os

def save_df_to_json(df, output_path):
    # convert dataframe -> list of dict
    data = df.to_dict(orient="records")

    # tạo folder nếu chưa có
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # lưu file json
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(data)} records to {output_path}")

df_clean = df_clean.fillna("")
# chạy
output_path = "../dataset/metadata_restaurant.json"
save_df_to_json(df_clean, output_path)

Saved 349 records to ../dataset/metadata_restaurant.json


In [29]:
check_meta = df_clean[["name", "url", "url_source", "canonical_url", "restaurant_id"]].copy()

print("Tổng số records:", len(check_meta))
print("Số restaurant_id duy nhất:", check_meta["restaurant_id"].nunique())
print("Số records bị dư:", len(check_meta) - check_meta["restaurant_id"].nunique())

dup_by_id = check_meta[check_meta.duplicated(subset=["restaurant_id"], keep=False)] \
    .sort_values(["restaurant_id", "name"])

print("\nSố dòng nằm trong nhóm bị trùng restaurant_id:", len(dup_by_id))
display(dup_by_id)

Tổng số records: 349
Số restaurant_id duy nhất: 349
Số records bị dư: 0

Số dòng nằm trong nhóm bị trùng restaurant_id: 0


,name,url,url_source,canonical_url,restaurant_id


In [3]:
df_reviews = pd.read_json("../dataset/preprocessed_q5/review_quan_5_preprocessed.json")
print("Số dòng review:", len(df_reviews))
print("Thiếu restaurant_id:", df_reviews["restaurant_id"].isna().sum())
df_reviews.head(2)

Số dòng review: 8369
Thiếu restaurant_id: 0


,restaurant_id,restaurant_url,canonical_url,restaurant_name,reviewer,rating,days_ago,comment,food_rating,service_rating,atmosphere_rating,comment_clean
0,e7b09f8c49e9533d4d79737ebd508a96,https://www.google.com/maps/place/Qu%C3%A1n+%C...,https://www.google.com/maps/place/Qu%C3%A1n+%C...,Quán ǎn Hùng Ký,Hai Thanh,4,1,"Đồ ăn ngon,đợi hơi lâu",4.0,3.0,4.0,"đồ ăn ngon, đợi hơi lâu"
1,e7b09f8c49e9533d4d79737ebd508a96,https://www.google.com/maps/place/Qu%C3%A1n+%C...,https://www.google.com/maps/place/Qu%C3%A1n+%C...,Quán ǎn Hùng Ký,Khiêm,3,28,"Đồ ăn khá ngon, mình đánh giá 9/10 điểm.\nQuán...",5.0,3.0,4.0,"đồ ăn khá ngon, mình đánh giá điểm, quán nằm t..."


In [4]:
import pandas as pd

df_meta = pd.read_json("../dataset/metadata_restaurant.json")
print("Số dòng metadata:", len(df_meta))
print("Thiếu restaurant_id:", df_meta["restaurant_id"].isna().sum())
df_meta.head(2)

Số dòng metadata: 349
Thiếu restaurant_id: 0


,url,name,avg_rating,review_count,category,price_range,address,phone,opening_hours,lgbtq_friendly,...,source_url_for_id,canonical_url,restaurant_id,price_min,price_max,price_level,rating,opening_parsed,popular_parsed,about_clean
0,https://www.google.com/maps/place/Qu%C3%A1n+%C...,Quán ǎn Hùng Ký,"3,3",102,Nhà hàng,100-200 N ₫,"Trần Hưng Đạo/218 B, Phường 11, Chợ Lớn, Hồ Ch...",+84 918 038 796,"{'Thứ Bảy': '17:00–23:00', 'Chủ Nhật': '17:00–...",unknown,...,https://www.google.com/maps/place/Qu%C3%A1n+%C...,https://www.google.com/maps/place/Qu%C3%A1n+%C...,e7b09f8c49e9533d4d79737ebd508a96,100000.0,200000.0,3.0,3.3,"{'sat': [[17, 0, 23, 0]], 'sun': [[17, 0, 23, ...",,"{'cac_tuy_chon_dich_vu': [], 'lua_chon_an_uong..."
1,https://www.google.com/maps/place/Van+Hoa+Rest...,Nhà hàng Văn Hoa,"4,3",801,Nhà hàng,,"68-76 Đ. Tản Đà, Phường 11, Chợ Lớn, Hồ Chí Mi...",+84 28 3856 3888,"{'Thứ Bảy': '07:00–13:30 17:00–21:30', 'Chủ Nh...",unknown,...,https://www.google.com/maps/place/Van+Hoa+Rest...,https://www.google.com/maps/place/Van+Hoa+Rest...,2904a482ed87ca8fbfa29425561d32bb,,,,4.3,"{'sat': [[7, 0, 13, 30], [17, 0, 21, 30]], 'su...","{'sun': {'6': 0, '7': 24, '8': 41, '9': 50, '1...","{'cac_tuy_chon_dich_vu': ['Ăn tại chỗ', 'Đồ ăn..."


In [5]:
meta_ids = set(df_meta["restaurant_id"].dropna().unique())
review_ids = set(df_reviews["restaurant_id"].dropna().unique())

print("Số quán trong metadata:", len(meta_ids))
print("Số quán trong review:", len(review_ids))
print("Số quán có trong review nhưng không có trong metadata:", len(review_ids - meta_ids))

Số quán trong metadata: 349
Số quán trong review: 341
Số quán có trong review nhưng không có trong metadata: 0
